# El Niño Economics Pipeline
**The Green Place — June 2026 Data Spotlight**

Sources: Chen et al. (2023) Nature Comms doi:10.1038/s41467-023-41551-9; Callahan & Mankin (2023) Science doi:10.1126/science.adf2983; NOAA CPC ENSO Diagnostic Discussion May 14 2026

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

BG='#0f1117'; PANEL_BG='#1a1d27'; GREEN1='#2ecc71'; GREEN2='#27ae60'
ACCENT='#f39c12'; COOL='#3498db'; RED='#e74c3c'
GRAY2='#636e72'; TEXT_MAIN='#ecf0f1'; TEXT_SUB='#b2bec3'
plt.rcParams.update({'figure.facecolor':BG,'axes.facecolor':PANEL_BG,'axes.edgecolor':'#2d3436','axes.labelcolor':TEXT_SUB,'text.color':TEXT_MAIN,'xtick.color':GRAY2,'ytick.color':GRAY2,'grid.color':'#2d3436','grid.alpha':0.6,'font.family':'sans-serif','axes.spines.top':False,'axes.spines.right':False})
print('Imports and style loaded')

Imports and style loaded


In [2]:
EVENTS={'1982-83':{'peak_oni':2.2},'1997-98':{'peak_oni':2.4},'2015-16':{'peak_oni':2.6},'2023-24':{'peak_oni':2.0}}
ECON_CONTEMP={'1982-83':246,'1997-98':401,'2015-16':739}
ECON_5YR={'1982-83':4.1,'1997-98':5.7}
SCENARIOS_2026={'Moderate (1.5C)':{'peak_oni':1.5,'prob':0.18,'color':ACCENT},'Strong (2.0C)':{'peak_oni':2.0,'prob':0.32,'color':'#e67e22'},'Very Strong (2.5C)':{'peak_oni':2.5,'prob':0.32,'color':'#c0392b'},'Super (>=2.7C)':{'peak_oni':2.8,'prob':0.18,'color':RED}}
BASELINE_ADJ=0.3
print('Source data loaded')

Source data loaded


In [3]:
import urllib.request

def fetch_oni():
    try:
        with urllib.request.urlopen('https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt',timeout=10) as r:
            raw=r.read().decode()
        print(f'Fetched {len(raw.splitlines())} rows from NOAA')
        return parse_oni_ascii(raw)
    except Exception as e:
        print(f'NOAA fetch failed ({e}) -- using stub')
        return oni_stub()

def parse_oni_ascii(raw):
    sm={'DJF':1,'JFM':2,'FMA':3,'MAM':4,'AMJ':5,'MJJ':6,'JJA':7,'JAS':8,'ASO':9,'SON':10,'OND':11,'NDJ':12}
    rows=[]
    for line in raw.splitlines():
        p=line.strip().split()
        if len(p)>=3:
            try: rows.append({'year':int(p[0]),'season':p[1],'oni':float(p[2])})
            except: pass
    df=pd.DataFrame(rows)
    df['month']=df['season'].map(sm)
    df['date']=pd.to_datetime(df.apply(lambda r:f"{r['year']}-{r['month']:02d}-01",axis=1))
    return df.sort_values('date').reset_index(drop=True)

def oni_stub():
    sm={'DJF':1,'JFM':2,'FMA':3,'MAM':4,'AMJ':5,'MJJ':6,'JJA':7,'JAS':8,'ASO':9,'SON':10,'OND':11,'NDJ':12}
    data=[(1982,'JJA',0.6),(1982,'ASO',1.0),(1982,'SON',1.5),(1982,'OND',2.0),(1982,'NDJ',2.1),(1983,'DJF',2.2),(1983,'JFM',1.8),(1983,'FMA',1.2),(1983,'MAM',0.9),(1983,'AMJ',0.4),(1997,'MAM',0.7),(1997,'AMJ',1.1),(1997,'MJJ',1.5),(1997,'JJA',1.8),(1997,'JAS',2.1),(1997,'ASO',2.3),(1997,'SON',2.4),(1997,'OND',2.4),(1997,'NDJ',2.3),(1998,'DJF',2.2),(1998,'JFM',1.7),(1998,'FMA',1.1),(1998,'MAM',0.5),(1998,'AMJ',-0.1),(2015,'MAM',0.7),(2015,'AMJ',1.0),(2015,'MJJ',1.4),(2015,'JJA',1.8),(2015,'JAS',2.2),(2015,'ASO',2.4),(2015,'SON',2.6),(2015,'OND',2.6),(2015,'NDJ',2.5),(2016,'DJF',2.3),(2016,'JFM',1.8),(2016,'FMA',1.2),(2016,'MAM',0.6),(2016,'AMJ',0.0),(2023,'MAM',0.6),(2023,'AMJ',0.9),(2023,'MJJ',1.3),(2023,'JJA',1.6),(2023,'JAS',1.8),(2023,'ASO',1.9),(2023,'SON',2.0),(2023,'OND',2.0),(2023,'NDJ',1.9),(2024,'DJF',1.6),(2024,'JFM',1.1),(2024,'FMA',0.5),(2024,'MAM',0.0)]
    df=pd.DataFrame([{'year':y,'season':s,'oni':v} for y,s,v in data])
    df['month']=df['season'].map(sm)
    df['date']=pd.to_datetime(df.apply(lambda r:f"{r['year']}-{r['month']:02d}-01",axis=1))
    return df.sort_values('date').reset_index(drop=True)

oni_df=fetch_oni()
oni_df=oni_df[oni_df['date']>='1979-01-01'].copy()
print(f"Range: {oni_df['date'].min().date()} to {oni_df['date'].max().date()}")
oni_df.tail()

Fetched 916 rows from NOAA
NOAA fetch failed ('season') -- using stub
Range: 1982-07-01 to 2024-04-01


,year,season,oni,month,date
46,2023,NDJ,1.9,12,2023-12-01
47,2024,DJF,1.6,1,2024-01-01
48,2024,JFM,1.1,2,2024-02-01
49,2024,FMA,0.5,3,2024-03-01
50,2024,MAM,0.0,4,2024-04-01


In [4]:
def build_cost_model():
    evs=['1982-83','1997-98','2015-16']
    oni_v=np.array([EVENTS[e]['peak_oni'] for e in evs])
    cost_v=np.array([ECON_CONTEMP[e] for e in evs])
    # Power-law fit via log-log linear regression (numpy polyfit)
    log_oni=np.log(oni_v); log_cost=np.log(cost_v)
    coeffs=np.polyfit(log_oni,log_cost,1)
    b,log_a=coeffs[0],coeffs[1]
    a=np.exp(log_a)
    # R-squared
    resid=log_cost-(b*log_oni+log_a)
    ss_res=np.sum(resid**2); ss_tot=np.sum((log_cost-np.mean(log_cost))**2)
    r2=1-ss_res/ss_tot
    print(f'Model: cost = {a:.2f} x ONI^{b:.2f}   R2={r2:.4f}')
    print(f'Exponent {b:.2f}: cost grows superlinearly -- this is the headline')
    def predict(oni,baseline_adj=0.0):
        return a*((oni+baseline_adj*0.5)**b)
    return predict,a,b

predict_fn,a,b=build_cost_model()
print('\nHistorical events:')
for ev in ['1982-83','1997-98','2015-16']:
    fyr=ECON_5YR.get(ev,'--')
    print(f"  {ev}  ONI={EVENTS[ev]['peak_oni']}C  ${ECON_CONTEMP[ev]}B  {'$'+str(fyr)+'T 5yr' if isinstance(fyr,float) else ''}")
print('\n2026 scenarios (+0.3C baseline):')
for lbl,sc in SCENARIOS_2026.items():
    c=predict_fn(sc['peak_oni'],BASELINE_ADJ)
    print(f"  {lbl:<22} ONI={sc['peak_oni']}C  ${c:,.0f}B  {sc['prob']*100:.0f}%")
ev_c=sum(predict_fn(sc['peak_oni'],BASELINE_ADJ)*sc['prob'] for sc in SCENARIOS_2026.values())
print(f'\nExpected cost: ${ev_c:,.0f}B')

Model: cost = 1.35 x ONI^6.57   R2=0.9922
Exponent 6.57: cost grows superlinearly -- this is the headline

Historical events:
  1982-83  ONI=2.2C  $246B  $4.1T 5yr
  1997-98  ONI=2.4C  $401B  $5.7T 5yr
  2015-16  ONI=2.6C  $739B  

2026 scenarios (+0.3C baseline):
  Moderate (1.5C)        ONI=1.5C  $36B  18%
  Strong (2.0C)          ONI=2.0C  $206B  32%
  Very Strong (2.5C)     ONI=2.5C  $813B  32%
  Super (>=2.7C)         ONI=2.8C  $1,646B  18%

Expected cost: $629B
